# Lab: From Linear Regression to Inverse Problems

**Master Smart Data, ENSAI** · Duration: 2 hours

This lab follows one thread. You first **fit** models to data: linear regression (OLS) and multiclass logistic regression.
You then see that a model with too much freedom can **overfit** or become **unstable**, and you fix it with **Ridge regularization**.
Finally, you discover that the very same idea solves a problem that looks unrelated at first sight: **recovering a sharp image from a blurry one**.

| Part | Topic | Data | Time |
|---|---|---|---|
| 0 | Environment check and reminders | | 5 min |
| 1 | Ordinary least squares | `diabetes` | 20 min |
| 2 | Multiclass logistic regression | `digits` | 30 min |
| 3 | Ridge regression | `diabetes` | 25 min |
| 4 | Introduction to inverse problems | an image | 30 min |

Before starting, you must have installed the lab environment by following the `README.md` of the repository (about 10 minutes).

**How to read this notebook**

* ✏️ **Paper question**: a short derivation to do with pen and paper.
* 💻 **Coding task**: complete the function or the line marked `# TODO`. Read the docstring first: it gives the inputs, the outputs and the formula.
* ✅ **Test cell**: run it right after your code. It prints `OK` when your implementation is correct.
* 💬 **Question**: write a short answer (two or three sentences) in the cell below it.
* 🎁 **Bonus**: skip it if you are short on time, and come back to it at the end.

All the code you need fits in a few lines. Use `numpy` only (no `scikit-learn` models) in your own functions.

---
## Part 0. Environment check and reminders

Run the cell below first. It checks that this notebook runs inside the virtual environment you created, and that the required packages are installed.
If it raises an error, go back to the `README.md` (section *Troubleshooting*).

In [4]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python      :", sys.version.split()[0])
print("Interpreter :", sys.executable)

if sys.prefix == sys.base_prefix:
    raise RuntimeError(
        "This notebook is NOT running inside a virtual environment.\n"
        "Activate .venv in your terminal, then restart Jupyter from that terminal "
        "(or select the .venv interpreter in VS Code)."
    )

missing = []
for package in ["numpy", "matplotlib", "scikit-learn"]:
    try:
        print("%-12s: %s" % (package, version(package)))
    except PackageNotFoundError:
        missing.append(package)
if missing:
    raise RuntimeError("Missing packages: %s. Run: pip install -r requirements.txt" % ", ".join(missing))

print("\nEnvironment OK")

Python      : 3.9.12
Interpreter : /Users/thibaultpautrel/Downloads/TP1/lab-regression/.venv/bin/python
numpy       : 2.0.2
matplotlib  : 3.9.4
scikit-learn: 1.6.1

Environment OK


In [5]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, load_digits, load_sample_image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures

plt.rcParams["figure.dpi"] = 100
rng = np.random.default_rng(0)

### Notation

We observe $m$ examples $(x_i, y_i)$, $i = 1, \dots, m$, where $x_i \in \mathbb{R}^d$ is a vector of features and $y_i$ is the target.
We stack the examples in a **design matrix** $X$ and a target vector $y$:

$$
X = \begin{bmatrix} x_1^\top \\ \vdots \\ x_m^\top \end{bmatrix} \in \mathbb{R}^{m \times d},
\qquad
y = \begin{bmatrix} y_1 \\ \vdots \\ y_m \end{bmatrix} \in \mathbb{R}^{m}.
$$

A linear model predicts $\hat y = X\theta$, where $\theta \in \mathbb{R}^d$ is the parameter vector.
To include an **intercept** $\theta_0$, we add a first column of ones to $X$.

### Good practice

* **Train / test split.** We fit the model on a training set and measure its performance on a test set that the model has never seen. The test error estimates the error on new data.
* **Standardization.** Features often have very different scales. We center each feature and divide it by its standard deviation. The mean and the standard deviation are computed **on the training set only**, then applied to the other sets.

The helper functions below are provided.

In [6]:
def add_intercept(X):
    # Add a first column of ones to X (shape (m, d) -> (m, d + 1)).
    return np.column_stack([np.ones(X.shape[0]), X])


def standardize(X_train, *X_others):
    # Standardize X_train, then apply the SAME transformation to the other arrays.
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    return [(X - mean) / std for X in (X_train,) + X_others]

---
## Part 1. Ordinary least squares (OLS)

### 1.1 Reminder

The linear model assumes $y = X\theta + \varepsilon$, where $\varepsilon$ is a noise term.
The OLS estimator minimizes the sum of squared residuals:

$$
\hat\theta_{\mathrm{OLS}} = \arg\min_{\theta} \|y - X\theta\|_2^2 .
$$

Setting the gradient $-2X^\top(y - X\theta)$ to zero gives the **normal equations**

$$
X^\top X \, \theta = X^\top y ,
$$

so that $\hat\theta_{\mathrm{OLS}} = (X^\top X)^{-1} X^\top y$ when $X^\top X$ is invertible.

**Numerical tip.** Never compute the inverse explicitly. Solve the linear system with `np.linalg.solve(A, b)`, which returns the solution of $A\theta = b$. It is faster and more accurate.

We measure the quality of a prediction $\hat y$ with two metrics:

$$
\mathrm{MSE} = \frac{1}{m}\sum_{i=1}^m (y_i - \hat y_i)^2,
\qquad
R^2 = 1 - \frac{\sum_i (y_i - \hat y_i)^2}{\sum_i (y_i - \bar y)^2}.
$$

The $R^2$ is the fraction of the variance of $y$ explained by the model. $R^2 = 1$ is a perfect fit. $R^2 = 0$ means the model does no better than predicting the mean $\bar y$.

### 1.2 The data

The `diabetes` dataset describes $m = 442$ patients with $d = 10$ baseline measurements:
age, sex, body mass index (`bmi`), average blood pressure (`bp`) and six blood serum measurements (`s1` to `s6`).
The target is a quantitative measure of **disease progression one year later**.

In [7]:
diabetes = load_diabetes(scaled=False)
X_diab, y_diab = diabetes.data, diabetes.target
feature_names = diabetes.feature_names

print("X shape:", X_diab.shape)
print("Features:", feature_names)
print("Target range:", y_diab.min(), "to", y_diab.max())

X shape: (442, 10)
Features: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
Target range: 25.0 to 346.0


### 1.3 💻 Coding task: OLS and metrics

In [ ]:
def ols_fit(X, y):
    """
    Fit a linear model by ordinary least squares.

    Parameters
    ----------
    X : array of shape (m, d)
        Design matrix (add the intercept column BEFORE calling this function).
    y : array of shape (m,)
        Targets.

    Returns
    -------
    theta : array of shape (d,)
        Solution of the normal equations  X^T X theta = X^T y.
        Use np.linalg.solve.
    """
    # TODO (one line)
    raise NotImplementedError


def mse(y_true, y_pred):
    """Mean squared error: (1/m) * sum_i (y_true_i - y_pred_i)^2."""
    # TODO (one line)
    raise NotImplementedError


def r2_score(y_true, y_pred):
    """Coefficient of determination: 1 - sum (y - y_pred)^2 / sum (y - mean(y))^2."""
    # TODO (one line)
    raise NotImplementedError

In [ ]:
# ✅ Test cell
X_t = add_intercept(rng.normal(size=(20, 3)))
y_t = rng.normal(size=20)
assert np.allclose(ols_fit(X_t, y_t), np.linalg.lstsq(X_t, y_t, rcond=None)[0])
assert np.isclose(mse(np.array([1., 2., 3.]), np.array([1., 2., 5.])), 4 / 3)
assert np.isclose(r2_score(y_t, y_t), 1.0)
assert np.isclose(r2_score(y_t, np.full(20, y_t.mean())), 0.0)
print("OK")

### 1.4 Simple regression: progression versus BMI

We start with a single feature, the body mass index. The model is $y = \theta_0 + \theta_1 \cdot \mathrm{bmi}$.

In [ ]:
bmi = X_diab[:, 2]
theta_simple = ols_fit(add_intercept(bmi[:, None]), y_diab)
print("theta_0 = %.1f,  theta_1 = %.2f" % tuple(theta_simple))

grid = np.linspace(bmi.min(), bmi.max(), 100)
plt.figure(figsize=(6, 4))
plt.scatter(bmi, y_diab, s=12, alpha=0.6, label="patients")
plt.plot(grid, theta_simple[0] + theta_simple[1] * grid, c="crimson", lw=2, label="OLS fit")
plt.xlabel("Body mass index")
plt.ylabel("Disease progression")
plt.legend()
plt.show()

print("R^2 = %.3f" % r2_score(y_diab, add_intercept(bmi[:, None]) @ theta_simple))

💬 **Question 1.1.** Interpret $\theta_1$ in plain words. What fraction of the variance of the progression does BMI alone explain?

*Your answer:*

### 1.5 Multiple regression: all ten features

We now use the ten features. We split the data (75 % train, 25 % test) and standardize the features.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_diab, y_diab, test_size=0.25, random_state=0)
X_tr_s, X_te_s = standardize(X_tr, X_te)
print("Train:", X_tr_s.shape, " Test:", X_te_s.shape)

💻 **Coding task.** Fit OLS on the standardized training set (with intercept). Compute the predictions on both sets and print the train and test $R^2$.

In [ ]:
theta_full = ...   # TODO: fit OLS with intercept on X_tr_s
y_tr_pred = ...    # TODO: predictions on the training set
y_te_pred = ...    # TODO: predictions on the test set

print("Train R^2 = %.3f" % r2_score(y_tr, y_tr_pred))
print("Test  R^2 = %.3f" % r2_score(y_te, y_te_pred))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].scatter(y_te, y_te_pred, s=15)
lims = [y_diab.min(), y_diab.max()]
ax[0].plot(lims, lims, "k--", lw=1)
ax[0].set_xlabel("Observed progression")
ax[0].set_ylabel("Predicted progression")
ax[0].set_title("Test set")

ax[1].bar(feature_names, theta_full[1:], color="steelblue")
ax[1].axhline(0, c="k", lw=0.8)
ax[1].set_title("OLS coefficients (standardized features)")
plt.show()

💬 **Question 1.2.** Why must the features be standardized before comparing the sizes of the coefficients?
Which features seem to matter most? Look at `s1` and `s2`: their coefficients are large and of opposite signs. Can you guess why?
(Hint: compute `np.corrcoef(X_diab[:, 4], X_diab[:, 5])`.)

💬 **Question 1.3.** Compare the train and test $R^2$. Does the model overfit?

*Your answer:*

---
## Part 2. Multiclass logistic regression

### 2.1 Reminder: binary logistic regression

When $y \in \{0, 1\}$, logistic regression models the probability of class 1 as

$$
P(Y = 1 \mid x) = \sigma(x^\top\theta), \qquad \sigma(z) = \frac{1}{1 + e^{-z}} .
$$

The parameters maximize the likelihood of a Bernoulli model. Equivalently, they minimize the **cross-entropy**

$$
J(\theta) = -\frac{1}{m}\sum_{i=1}^m \Big[ y_i \log \sigma(x_i^\top\theta) + (1 - y_i)\log\big(1 - \sigma(x_i^\top\theta)\big) \Big],
$$

whose gradient is $\nabla J(\theta) = \frac{1}{m} X^\top\big(\sigma(X\theta) - y\big)$.

### 2.2 From 2 classes to $K$ classes: the softmax model

Now $y \in \{0, 1, \dots, K-1\}$. We give each class $k$ its own parameter vector $\theta_k \in \mathbb{R}^d$ and compute a **score** $z_k = x^\top\theta_k$.
The **softmax** function turns the $K$ scores into probabilities:

$$
P(Y = k \mid x) = \frac{e^{z_k}}{\sum_{j=0}^{K-1} e^{z_j}} .
$$

These numbers are positive and sum to 1. The predicted class is the one with the largest probability, which is also the one with the largest score.

**Matrix form.** We store the $\theta_k$ as the columns of a matrix $\Theta \in \mathbb{R}^{d \times K}$.
The matrix $Z = X\Theta \in \mathbb{R}^{m \times K}$ contains all the scores, one row per example.
Applying the softmax to each row gives the matrix of probabilities $P \in \mathbb{R}^{m \times K}$.

✏️ **Paper question 2.1.** Take $K = 2$. Show that $P(Y = 1 \mid x) = \sigma\big(x^\top(\theta_1 - \theta_0)\big)$.
Conclude that softmax regression with two classes is exactly binary logistic regression.

*Your answer:*

### 2.3 Cost and gradient

We encode the labels with **one-hot vectors**: $Y \in \{0,1\}^{m \times K}$ with $Y_{ik} = 1$ if $y_i = k$ and $0$ otherwise.
Maximizing the likelihood of the model (a categorical distribution, which generalizes the Bernoulli) amounts to minimizing the cross-entropy

$$
J(\Theta) = -\frac{1}{m}\sum_{i=1}^m \sum_{k=0}^{K-1} Y_{ik} \log P_{ik} .
$$

For each example, only the term of the true class survives: we penalize a low probability on the correct class.
The gradient has the same form as in the binary case (we admit it):

$$
\nabla_\Theta J = \frac{1}{m} X^\top (P - Y) \in \mathbb{R}^{d \times K}.
$$

There is no closed-form solution. We use **gradient descent**: start from $\Theta = 0$ and repeat

$$
\Theta \leftarrow \Theta - \eta \, \nabla_\Theta J(\Theta),
$$

where $\eta > 0$ is the step size (learning rate).

**Numerical tip.** $e^{z}$ overflows for large $z$. Since the softmax does not change when we subtract the same constant from all scores, we subtract the maximum score of each row before taking the exponential.

### 2.4 The data

The `digits` dataset contains 1797 images of handwritten digits, each of size $8 \times 8$ pixels, with values from 0 to 16.
Each image becomes a vector of $64$ features. There are $K = 10$ classes.

In [ ]:
digits = load_digits()
X_dig = digits.data / 16.0          # pixel values in [0, 1]
y_dig = digits.target
K = 10

fig, ax = plt.subplots(1, 10, figsize=(12, 1.6))
for k in range(10):
    ax[k].imshow(X_dig[k].reshape(8, 8), cmap="gray_r")
    ax[k].set_title(y_dig[k])
    ax[k].axis("off")
plt.show()

X_dig_tr, X_dig_te, y_dig_tr, y_dig_te = train_test_split(
    add_intercept(X_dig), y_dig, test_size=0.25, random_state=0)
print("Train:", X_dig_tr.shape, " Test:", X_dig_te.shape)

### 2.5 💻 Coding task: softmax regression

In [ ]:
def one_hot(y, K):
    """
    One-hot encoding.
    y : array of shape (m,) with integer labels in {0, ..., K-1}.
    Returns Y of shape (m, K) with Y[i, k] = 1 if y[i] == k, else 0.
    Hint: np.eye(K) is a K x K identity matrix, and np.eye(K)[y] selects its rows.
    """
    # TODO (one line)
    raise NotImplementedError


def softmax(Z):
    """
    Row-wise softmax.
    Z : array of shape (m, K) of scores.
    Returns P of shape (m, K) with P[i, k] = exp(Z[i, k]) / sum_j exp(Z[i, j]).
    Subtract the maximum of each row first (use keepdims=True) to avoid overflow.
    """
    # TODO (three lines)
    raise NotImplementedError


def cross_entropy(Theta, X, Y):
    """
    Cost J(Theta) = -(1/m) * sum_i sum_k Y[i, k] * log P[i, k],  with P = softmax(X @ Theta).
    Theta : (d, K),  X : (m, d),  Y : (m, K) one-hot.
    """
    # TODO (two lines)
    raise NotImplementedError


def gradient(Theta, X, Y):
    """
    Gradient of the cross-entropy:  (1/m) * X^T (P - Y),  with P = softmax(X @ Theta).
    Returns an array of shape (d, K).
    """
    # TODO (two lines)
    raise NotImplementedError


def predict_class(Theta, X):
    """Predicted labels: index of the largest score in each row of X @ Theta (use np.argmax)."""
    # TODO (one line)
    raise NotImplementedError

In [ ]:
# ✅ Test cell
assert np.array_equal(one_hot(np.array([2, 0]), 3), np.array([[0, 0, 1], [1, 0, 0]]))

P_t = softmax(np.array([[1., 2., 3.], [1000., 1000., 1000.]]))
assert np.allclose(P_t.sum(axis=1), 1), "Each row must sum to 1."
assert np.allclose(P_t[0], [0.09003057, 0.24472847, 0.66524096])
assert np.allclose(P_t[1], 1 / 3), "Did you subtract the row maximum?"

# Compare the gradient with a finite-difference approximation
X_t = rng.normal(size=(15, 4)); Y_t = one_hot(rng.integers(0, 3, 15), 3)
Theta_t = rng.normal(size=(4, 3)); E = np.zeros((4, 3)); E[1, 2] = 1e-6
num = (cross_entropy(Theta_t + E, X_t, Y_t) - cross_entropy(Theta_t - E, X_t, Y_t)) / 2e-6
assert np.isclose(gradient(Theta_t, X_t, Y_t)[1, 2], num, atol=1e-6)
assert np.array_equal(predict_class(np.eye(3), np.array([[0., 5., 1.]])), [1])
print("OK")

💻 **Coding task.** Complete the gradient descent loop.

In [ ]:
def fit_softmax(X, y, K, step=0.5, n_iter=1000):
    """
    Fit softmax regression by gradient descent, starting from Theta = 0.

    Returns
    -------
    Theta : array of shape (d, K)
    history : list of the cost values, one per iteration
    """
    Y = one_hot(y, K)
    Theta = np.zeros((X.shape[1], K))
    history = []
    for t in range(n_iter):
        history.append(cross_entropy(Theta, X, Y))
        Theta = ...   # TODO: one gradient descent step
    return Theta, history

In [ ]:
Theta_dig, history = fit_softmax(X_dig_tr, y_dig_tr, K)

plt.figure(figsize=(6, 3.5))
plt.plot(history)
plt.xlabel("Iteration")
plt.ylabel("Cross-entropy (train)")
plt.yscale("log")
plt.show()

### 2.6 Evaluation

💻 **Coding task.** Compute the train and test accuracies (fraction of correctly classified images).

In [ ]:
acc_tr = ...   # TODO
acc_te = ...   # TODO
print("Train accuracy: %.3f" % acc_tr)
print("Test  accuracy: %.3f" % acc_te)

The **confusion matrix** $C$ counts, for each pair of classes $(k, l)$, the number of test images of true class $k$ predicted as class $l$.
A perfect classifier has a diagonal confusion matrix.

In [ ]:
y_hat = predict_class(Theta_dig, X_dig_te)
C = np.zeros((K, K), dtype=int)
np.add.at(C, (y_dig_te, y_hat), 1)

fig, ax = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1, 1.3]})
ax[0].imshow(C, cmap="Blues")
for i in range(K):
    for j in range(K):
        ax[0].text(j, i, C[i, j], ha="center", va="center", fontsize=8,
                   color="white" if C[i, j] > C.max() / 2 else "black")
ax[0].set_xticks(range(K)); ax[0].set_yticks(range(K))
ax[0].set_xlabel("Predicted class"); ax[0].set_ylabel("True class")
ax[0].set_title("Confusion matrix (test)")

wrong = np.where(y_hat != y_dig_te)[0][:8]
ax[1].axis("off")
for j, i in enumerate(wrong):
    a = fig.add_axes([0.52 + 0.11 * (j % 4), 0.5 - 0.38 * (j // 4), 0.09, 0.3])
    a.imshow(X_dig_te[i, 1:].reshape(8, 8), cmap="gray_r")
    a.set_title("true %d / pred %d" % (y_dig_te[i], y_hat[i]), fontsize=8)
    a.axis("off")
ax[1].set_title("Some misclassified test images")
plt.show()

💬 **Question 2.2.** Which digits does the model confuse? Do the errors look understandable to you?

*Your answer:*

### 2.7 What did the model learn?

Each column $\theta_k$ contains one weight per pixel (plus the intercept). We can display the 64 pixel weights as an $8 \times 8$ image.
A **red** pixel has a positive weight: ink at this place increases the score of class $k$.
A **blue** pixel has a negative weight: ink at this place decreases it.
We compare these weights with the average image of each class.

In [ ]:
fig, ax = plt.subplots(2, 10, figsize=(14, 3.2))
for k in range(K):
    ax[0, k].imshow(X_dig[y_dig == k].mean(axis=0).reshape(8, 8), cmap="gray_r")
    ax[0, k].set_title(k)
    w = Theta_dig[1:, k]
    ax[1, k].imshow(w.reshape(8, 8), cmap="RdBu_r", vmin=-np.abs(w).max(), vmax=np.abs(w).max())
for a in ax.ravel():
    a.axis("off")
ax[0, 0].text(-6, 4, "mean\nimage", fontsize=9, va="center")
ax[1, 0].text(-6, 4, "weights", fontsize=9, va="center")
plt.show()

💬 **Question 2.3.** Look at the weights of the digits 0, 1 and 3. Explain what the model "looks for" in an image of each class.

*Your answer:*

---
## Part 3. Ridge regression

### 3.1 Motivation: when OLS breaks down

In Part 1, we had 10 features and 331 training patients. OLS worked well.
Things change when the number of features gets close to the number of examples.
To see it, we enrich the `diabetes` data with all the **products of pairs of features** (`age*bmi`, `bmi^2`, ...).
This gives 64 features. We also keep only **100 patients** for training.
The other patients are split into a **validation set** (to tune hyperparameters) and a **test set** (for the final evaluation).

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_diab)
poly_names = poly.get_feature_names_out(feature_names)

# 'sex' takes only two values, so 'sex^2' is an exact copy of 'sex' (up to an affine map): we drop it.
keep = poly_names != "sex^2"
X_poly, poly_names = X_poly[:, keep], poly_names[keep]

X_p_tr, X_rest, y_p_tr, y_rest = train_test_split(X_poly, y_diab, train_size=100, random_state=1)
X_p_va, X_p_te, y_p_va, y_p_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=1)
X_p_tr, X_p_va, X_p_te = standardize(X_p_tr, X_p_va, X_p_te)
print("Train:", X_p_tr.shape, " Validation:", X_p_va.shape, " Test:", X_p_te.shape)

In [ ]:
theta_ols = ols_fit(add_intercept(X_p_tr), y_p_tr)
print("OLS  train MSE = %8.0f" % mse(y_p_tr, add_intercept(X_p_tr) @ theta_ols))
print("OLS  test  MSE = %8.0f" % mse(y_p_te, add_intercept(X_p_te) @ theta_ols))
print("Largest |coefficient| = %.0f" % np.abs(theta_ols[1:]).max())

💬 **Question 3.1.** Compare the train and test errors. How large are the coefficients? What is happening?

*Your answer:*

### 3.2 Reminder: Ridge regression

Ridge adds a penalty on the size of the coefficients:

$$
\hat\theta_\lambda = \arg\min_{\theta} \; J(\theta) = \|y - X\theta\|_2^2 + \lambda \|\theta\|_2^2, \qquad \lambda > 0 .
$$

The parameter $\lambda$ sets the trade-off between fitting the data and keeping the coefficients small.

**About the intercept.** We do not want to penalize the intercept. A simple way is to work with centered data: the features are standardized, and we subtract the training mean $\bar y$ from the target.
The intercept is then $\bar y$, and Ridge is applied without intercept.

✏️ **Paper question 3.2.**

1. Compute the gradient $\nabla J(\theta)$. Deduce that $\hat\theta_\lambda$ satisfies $(X^\top X + \lambda I_d)\,\theta = X^\top y$.
2. Compute the Hessian $\nabla^2 J(\theta)$. Why is $J$ strictly convex?
3. Show that $X^\top X + \lambda I_d$ is invertible for every $\lambda > 0$, even when $X^\top X$ is not. (Hint: compute $v^\top (X^\top X + \lambda I_d) v$.)
4. What does $\hat\theta_\lambda$ become when $\lambda \to 0$? When $\lambda \to +\infty$?

*Your answer:*

### 3.3 💻 Coding task: Ridge

In [ ]:
def ridge_fit(X, y, lam):
    """
    Ridge regression WITHOUT intercept.

    Parameters
    ----------
    X : array of shape (m, d)
    y : array of shape (m,)
    lam : float, regularization parameter lambda > 0

    Returns
    -------
    theta : array of shape (d,)
        Solution of  (X^T X + lam * I) theta = X^T y.
        Use np.linalg.solve and np.eye.
    """
    # TODO (two lines)
    raise NotImplementedError

In [ ]:
# ✅ Test cell
from sklearn.linear_model import Ridge   # used only to check your result
X_t = rng.normal(size=(20, 30)); y_t = rng.normal(size=20)   # more features than examples
assert np.allclose(ridge_fit(X_t, y_t, 0.7), Ridge(alpha=0.7, fit_intercept=False).fit(X_t, y_t).coef_)
print("OK")

### 3.4 Choosing $\lambda$

💻 **Coding task.** For each $\lambda$ in a logarithmic grid, fit Ridge on the centered training targets, then store the training and validation MSE.
Remember that the prediction is $\bar y + X\theta$, where $\bar y$ is the training mean.

In [ ]:
lambdas = np.logspace(-5, 4, 40)
y_bar = y_p_tr.mean()
train_err, val_err, coefs = [], [], []

for lam in lambdas:
    theta = ...                  # TODO: Ridge on the centered training targets
    coefs.append(theta)
    train_err.append(...)        # TODO: training MSE
    val_err.append(...)          # TODO: validation MSE
coefs = np.array(coefs)

In [ ]:
lam_best = lambdas[np.argmin(val_err)]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].plot(lambdas, train_err, label="train")
ax[0].plot(lambdas, val_err, label="validation")
ax[0].axvline(lam_best, c="gray", ls="--", label=r"best $\lambda$")
ax[0].set_xscale("log"); ax[0].set_yscale("log")
ax[0].set_xlabel(r"$\lambda$"); ax[0].set_ylabel("MSE")
ax[0].set_title("Error versus regularization")
ax[0].legend()

ax[1].plot(lambdas, coefs, lw=0.8)
ax[1].axvline(lam_best, c="gray", ls="--")
ax[1].set_xscale("log")
ax[1].set_xlabel(r"$\lambda$"); ax[1].set_ylabel("coefficient value")
ax[1].set_title("Regularization path (one curve per feature)")
plt.show()

theta_best = ridge_fit(X_p_tr, y_p_tr - y_bar, lam_best)
print("Best lambda      = %.1f" % lam_best)
print("Ridge test MSE   = %.0f" % mse(y_p_te, y_bar + X_p_te @ theta_best))
print("OLS   test MSE   = %.0f" % mse(y_p_te, add_intercept(X_p_te) @ theta_ols))

💬 **Question 3.3.** Describe the two curves of the left plot. Why does the training error always increase with $\lambda$ while the validation error first decreases?
Relate the two ends of the curve to the words *overfitting* and *underfitting*.

💬 **Question 3.4.** Describe the regularization path. What happens to the coefficients when $\lambda$ grows?

💬 **Question 3.5.** Why do we choose $\lambda$ on the validation set and not on the test set?

*Your answer:*

**Remark: regularized logistic regression.** The same penalty applies to the classification model of Part 2. One minimizes $J(\Theta) + \frac{\lambda}{2}\|\Theta\|^2$, and the gradient simply gains the term $\lambda\Theta$. This is what `scikit-learn` does by default in `LogisticRegression`.

---
## Part 4. Introduction to inverse problems

### 4.1 What is an inverse problem?

In Parts 1 to 3, we knew the inputs $X$ and the outputs $y$, and we looked for the parameters $\theta$ of a model.
In an **inverse problem**, we observe a degraded version of an object, and we want to recover the object itself.

* A photo is blurred because the camera moved or was out of focus. We want the sharp photo.
* A telescope image is blurred by the atmosphere. We want the true sky.
* A medical scanner does not measure the image of the body directly. It measures X-ray projections through the body. We want the image.

In all these cases, going **from the object to the measurement** is easy: physics tells us how. This is the *direct problem*.
Going **back from the measurement to the object** is hard, as you will see. This is the *inverse problem*.

We will deblur an image, and we will see that the tools of this lab (OLS and Ridge) are exactly what we need.

### 4.2 Making a blurry image

We load a grayscale image of size $48 \times 48$, with values between 0 (black) and 1 (white).
We see it as a vector $x \in \mathbb{R}^{n}$ with $n = 48^2 = 2304$ pixels.

In [ ]:
def load_image(size=48):
    # Grayscale image of shape (size, size) with values in [0, 1].
    img = load_sample_image("china.jpg").mean(axis=2) / 255.0
    img = img[:420, :420]                       # square crop around the pagoda
    f = 420 // size
    img = img[:f * size, :f * size]
    return img.reshape(size, f, size, f).mean(axis=(1, 3))   # downsampling


def show_images(images, titles, size=48):
    # Display a list of images (vectors or 2D arrays) side by side, gray levels clipped to [0, 1].
    fig, ax = plt.subplots(1, len(images), figsize=(3 * len(images), 3.2))
    for a, im, t in zip(np.atleast_1d(ax), images, titles):
        a.imshow(np.reshape(im, (size, size)), cmap="gray", vmin=0, vmax=1)
        a.set_title(t, fontsize=10)
        a.axis("off")
    plt.show()


x_true = load_image().ravel()
n = x_true.size
show_images([x_true], ["original image x"])

**Blurring is a linear operation.** Blurring replaces each pixel by a weighted average of its neighbors: close neighbors get a large weight, far ones a small weight.
Each blurred pixel is therefore a linear combination of the pixels of $x$. So the blurred image is $Ax$ for some matrix $A \in \mathbb{R}^{n \times n}$.
Row $i$ of $A$ contains the weights used to compute pixel $i$.

The function below builds $A$. You do not need to understand its construction. Look instead at one row of $A$, displayed as an image.

In [ ]:
def make_blur_matrix(size=48, width=1.5):
    # Gaussian blur matrix of shape (size^2, size^2). 'width' is the blur radius in pixels.
    i = np.arange(size)
    B = np.exp(-(i[:, None] - i[None, :]) ** 2 / (2 * width ** 2))
    B = B / B.sum(axis=1, keepdims=True)         # 1D blur along one axis
    return np.kron(B, B)                          # 2D blur = blur rows and columns


A = make_blur_matrix()
print("A has shape", A.shape)

center = 24 * 48 + 24                             # index of a pixel near the center
plt.figure(figsize=(3.5, 3.5))
plt.imshow(A[center].reshape(48, 48)[14:35, 14:35], cmap="viridis")
plt.title("Row of A for a central pixel\n(zoom): blur weights")
plt.axis("off")
plt.show()

A real measurement is never perfect. We add a small random **noise** $\varepsilon$:

$$
y = A x + \varepsilon .
$$

This is a linear model, as in Part 1. But the roles change: $A$ is known, $y$ is observed, and the **unknown is the image $x$**.

💻 **Coding task.** Compute the blurred image `y_clean` $= Ax$ and the noisy observation `y` $= Ax + \varepsilon$, with $\varepsilon$ a Gaussian vector of standard deviation 0.01 (use `rng.normal`).

In [ ]:
rng = np.random.default_rng(0)   # fixed seed: everybody gets the same noise
noise_level = 0.01
y_clean = ...   # TODO
y = ...         # TODO

show_images([x_true, y_clean, y], ["original x", "blurred Ax", "observed y = Ax + noise"])

The noise is so small that you can hardly see it. Keep this in mind.

### 4.3 Why we cannot simply "undo" the blur

The matrix $A$ is square and invertible. The natural idea is to solve $Ax = y$.
Note that this is exactly the OLS estimator of Part 1, with $A$ in the role of the design matrix: $(A^\top A)^{-1}A^\top y = A^{-1}y$.

We also measure the quality of a reconstruction $\hat x$ with the relative error $\|\hat x - x\| / \|x\|$.

💻 **Coding task.** Solve the system with `np.linalg.solve`, first with `y_clean`, then with `y`.

In [ ]:
def relative_error(x_hat):
    return np.linalg.norm(x_hat - x_true) / np.linalg.norm(x_true)

In [ ]:
x_naive_clean = ...   # TODO
x_naive = ...         # TODO

print("Relative error without noise: %.2e" % relative_error(x_naive_clean))
print("Relative error with noise:    %.2e" % relative_error(x_naive))
show_images([y, x_naive_clean, x_naive],
            ["observed y", "solve(A, Ax): no noise", "solve(A, y): with noise"])

💬 **Question 4.1.** Without noise, the image comes back perfectly. With a noise you could not even see, the result is useless. How large are the values of `x_naive` compared with the gray levels of an image (between 0 and 1)? (Try `np.abs(x_naive).max()`.)

*Your answer:*

**Where does this amplification come from?** Let us blur two simple patterns: wide stripes (slow variations) and very fine stripes (fast variations, one pixel wide).

In [ ]:
rows = np.arange(48)
wide = np.tile(0.5 + 0.5 * np.sin(2 * np.pi * rows / 16), (48, 1)).T.ravel()
fine = np.tile(0.5 + 0.5 * (-1.0) ** rows, (48, 1)).T.ravel()

def contrast(v):
    # Amplitude of the variations, measured away from the borders.
    v = v.reshape(48, 48)[8:40, 8:40]
    return (v.max() - v.min()) / 2

show_images([wide, A @ wide, fine, A @ fine],
            ["wide stripes", "wide stripes, blurred", "fine stripes", "fine stripes, blurred"])
print("Contrast kept by the blur, wide stripes: %.3f" % (contrast(A @ wide) / contrast(wide)))
print("Contrast kept by the blur, fine stripes: %.6f" % (contrast(A @ fine) / contrast(fine)))

The blur barely changes slow variations. It almost **erases fine details**: the contrast of the fine stripes is divided by about 30 000. A checkerboard, which is fine in both directions, is even divided by about one billion.

To undo the blur, the inversion must therefore **multiply the fine details by these huge factors**. On a clean image, this is fine: it restores exactly what the blur removed.
But the noise contains fine details too, as random pixel-to-pixel fluctuations. They are multiplied by the same huge factors. A noise of 0.01 becomes fluctuations of thousands or millions. This is why the naive reconstruction above looks like a checkerboard.

This is the core difficulty of inverse problems: **the solution is extremely sensitive to small errors in the data**.
Notice the link with Part 3: there, correlated features made $X^\top X$ nearly singular and the OLS coefficients exploded. Here, $A$ is nearly singular and the OLS image explodes.

🎁 *Bonus, for those who know the SVD:* the fine stripes are close to singular vectors of $A$ with tiny singular values. Inverting $A$ divides by these singular values.

### 4.4 The fix: Ridge regularization

We know a cure for an unstable OLS estimator: Ridge. Instead of solving $Ax = y$ exactly, we look for

$$
\hat x_\lambda = \arg\min_{x} \|y - Ax\|_2^2 + \lambda \|x\|_2^2 .
$$

The first term asks $\hat x$ to explain the observation. The second term forbids the huge values that the noise created.
This is **exactly** your function `ridge_fit`, with $A$ in place of $X$ and the image $x$ in place of $\theta$.
In the field of inverse problems, this method is called **Tikhonov regularization**, after the mathematician who introduced it in the 1940s.

💻 **Coding task.** Compute the Ridge reconstructions for the values of $\lambda$ below. (Each one takes about a second.)

In [ ]:
lambdas_img = [1e-5, 1e-3, 1e-2, 1e-1, 1.0]
x_ridge = ...   # TODO: list of the Ridge reconstructions, one per lambda

show_images([y] + x_ridge, ["observed y"] + [r"$\lambda$ = %g" % lam for lam in lambdas_img])
for lam, xr in zip(lambdas_img, x_ridge):
    print("lambda = %-7g  relative error = %.3f" % (lam, relative_error(xr)))
print("Error of the blurred image itself: %.3f" % relative_error(y))

🎁 **Bonus.** Here we know the true image, so we can draw the full error curve. (This cell takes a few seconds.)

In [ ]:
lambda_grid = np.logspace(-6, 1, 12)
errors = [relative_error(ridge_fit(A, y, lam)) for lam in lambda_grid]

plt.figure(figsize=(6, 3.8))
plt.plot(lambda_grid, errors, "o-")
plt.axhline(relative_error(y), c="gray", ls="--", label="blurred image y")
plt.xscale("log"); plt.yscale("log")
plt.xlabel(r"$\lambda$"); plt.ylabel("relative error")
plt.title("Reconstruction error versus regularization")
plt.legend()
plt.show()

💬 **Question 4.2.** Describe the reconstructions when $\lambda$ is too small and when $\lambda$ is too large. Compare the error curve with the validation curve of Part 3.

💬 **Question 4.3.** For large $\lambda$, the image gets darker. Why? (Hint: towards which image does the penalty $\|x\|^2$ pull $\hat x$?)

💬 **Question 4.4.** Does the best reconstruction contain more details than the observed image $y$? Which details are still missing?

💬 **Question 4.5.** Here we chose $\lambda$ using the true image. In a real application, the true image is unknown. Suggest an idea to choose $\lambda$.

*Your answer:*

### 4.5 Where is the research today?

*Reading at home if you are short on time.*

The Ridge penalty encodes a very weak knowledge about the image: "its pixel values are not too large". It ignores everything we know about what a real image looks like: smooth areas, sharp edges, repeated textures.
Better reconstructions come from **better knowledge of what a realistic image is**.

* **Classical approaches** (1990s to 2010s) design this knowledge by hand. For example, they assume that an image has few edges, or that it can be described with few coefficients in a well-chosen basis.
* **Current approaches** *learn* this knowledge from large collections of images with neural networks. A network trained to denoise images can be plugged into the reconstruction algorithm in place of the penalty ("plug-and-play" methods). Generative models, such as the diffusion models used to create images, can also guide the reconstruction towards realistic images.

These ideas have direct applications:
**MRI** scans that are several times faster, **CT** scans with a lower radiation dose, and the first **image of a black hole** (M87, Event Horizon Telescope, 2019), reconstructed from very incomplete radio measurements.

The common thread with this lab stays the same: fit the data, but not too closely, using what you know about the solution.

---
## Summary

| Model | Problem solved | Key formula |
|---|---|---|
| OLS | fit a linear model | $X^\top X\,\theta = X^\top y$ |
| Softmax regression | classify into $K$ classes | $\nabla J = \frac{1}{m}X^\top(P - Y)$, gradient descent |
| Ridge | stabilize OLS, reduce overfitting | $(X^\top X + \lambda I)\,\theta = X^\top y$ |
| Tikhonov | recover an image from blurred, noisy data | Ridge with $A$ in place of $X$ |

The same choice appears everywhere: a **trade-off** between fidelity to the data and stability of the solution, controlled by $\lambda$.